In [ ]:
import pandas as pd
import altair as alt

from google.colab import files
uploaded = files.upload()

df = pd.read_csv('comercio_cultural_limpia.csv')
columnas_anios = [c for c in df.columns if 'Total en USD' in c]

df_long = df.melt(
    id_vars=['País', 'Producto', 'Flujo', 'Destino'],
    value_vars=columnas_anios,
    var_name='Columna_Anio',
    value_name='USD_Millones'
)

df_long['Año'] = df_long['Columna_Anio'].str.extract(r'(\d{4})').astype(int)
df_long['Fecha'] = pd.to_datetime(df_long['Año'].astype(str) + '-01-01')

china_sw = df_long[
    (df_long['País'] == 'China') &
    (df_long['Producto'] == 'Software, videojuegos y recorded media')
].copy()

china_mundial = china_sw[china_sw['Destino'] == 'Mundial'].copy()

china_destinos = china_sw[
    (china_sw['Flujo'] == 'Exportación') &
    (china_sw['Destino'] != 'Mundial')
].copy()

hitos = pd.DataFrame([
    {'Fecha': pd.to_datetime('2007-01-01'), 'Año': 2007, 'USD_Millones': 9365,
     'Hito': '2007: 1° cruce a exportador neto ($9.365M vs $8.786M)'},
    {'Fecha': pd.to_datetime('2011-01-01'), 'Año': 2011, 'USD_Millones': 12269,
     'Hito': '2011: Último déficit comercial (-$1.787M)'},
    {'Fecha': pd.to_datetime('2012-01-01'), 'Año': 2012, 'USD_Millones': 19185,
     'Hito': '2012: Inflexión definitiva (+83% anual, $19.185M)'},
    {'Fecha': pd.to_datetime('2018-01-01'), 'Año': 2018, 'USD_Millones': 23850,
     'Hito': '2018: Pico pre-pandemia ($23.850M) y récord Imp. ($14.387M)'},
    {'Fecha': pd.to_datetime('2022-01-01'), 'Año': 2022, 'USD_Millones': 29532,
     'Hito': '2022: Récord histórico de exportación ($29.532M)'}
])

lineas_globales = alt.Chart(china_mundial).mark_line(
    point=alt.OverlayMarkDef(size=65, filled=True),
    strokeWidth=3
).encode(
    x=alt.X('year(Fecha):T', title='Año', axis=alt.Axis(format='%Y', tickCount=23)),
    y=alt.Y('USD_Millones:Q', title='Millones de USD', scale=alt.Scale(domain=[0, 33000])),
    color=alt.Color(
        'Flujo:N',
        title='Flujo Global (China)',
        scale=alt.Scale(domain=['Exportación', 'Importación'], range=['#d95f02', '#1b9e77'])
    ),
    tooltip=[
        alt.Tooltip('Año:O', title='Año'),
        alt.Tooltip('Flujo:N', title='Flujo'),
        alt.Tooltip('USD_Millones:Q', title='Millones de USD', format=',.0f')
    ]
)

reglas_hitos = alt.Chart(hitos).mark_rule(
    color='#666666', strokeDash=[4, 4], strokeWidth=1.2
).encode(
    x='year(Fecha):T'
)

puntos_hitos = alt.Chart(hitos).mark_point(
    color='#b30000', size=110, filled=True
).encode(
    x='year(Fecha):T',
    y='USD_Millones:Q',
    tooltip=[alt.Tooltip('Año:O'), alt.Tooltip('Hito:N', title='Evento Clave')]
)

textos_hitos = alt.Chart(hitos).mark_text(
    align='center', dy=-14, fontSize=10.5, fontWeight='bold', color='#222222'
).encode(
    x='year(Fecha):T',
    y='USD_Millones:Q',
    text='Hito:N'
)

panel_superior = (lineas_globales + reglas_hitos + puntos_hitos + textos_hitos).properties(
    width=820,
    height=320,
    title='China: De importador pasivo a principal exportador de software y videojuegos
)

seleccion_destino = alt.selection_multi(fields=['Destino'], bind='legend')

orden_destinos = ['Estados Unidos', 'Unión Europea', 'Japón', 'Reino Unido', 'República de Corea']

areas_destinos = alt.Chart(china_destinos).mark_area(
    line=True, point=True
).encode(
    x=alt.X('year(Fecha):T', title='Año', axis=alt.Axis(format='%Y', tickCount=23)),
    y=alt.Y('USD_Millones:Q', title='Exportaciones Bilaterales (Millones de USD)', stack='zero'),
    color=alt.Color(
        'Destino:N',
        title='Socio de destino (Clic para filtrar)',
        sort=orden_destinos,
        scale=alt.Scale(
            domain=orden_destinos,
            range=['#2b5c8f', '#e69f00', '#d55e00', '#56b4e9', '#009e73']
        )
    ),
    order=alt.Order('color_Destino_sort_index:Q'),
    opacity=alt.condition(seleccion_destino, alt.value(0.85), alt.value(0.15)),
    tooltip=[
        alt.Tooltip('Año:O', title='Año'),
        alt.Tooltip('Destino:N', title='País / Región de destino'),
        alt.Tooltip('USD_Millones:Q', title='Exportado (millones USD)', format=',.0f')
    ]
).add_selection(
    seleccion_destino
).properties(
    width=820,
    height=320,
    title='¿Hacia dónde exporta China?: Volumen de exportación por potencia'
)

dashboard_china = alt.vconcat(
    panel_superior,
    areas_destinos
).resolve_scale(
    color='independent'
).configure_title(
    fontSize=14,
    anchor='start',
    color='#1a1a1a'
).configure_axis(
    labelFontSize=11,
    titleFontSize=12
)

dashboard_china.save('flujos_china_software_interactivo.html')
dashboard_china